[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/%E6%A9%9F%E7%8E%87%E7%B5%B1%E8%A8%88/05_%E6%AD%A3%E6%85%8B%E5%88%86%E4%BD%88.ipynb) · [回章節首頁](./README.md)

## 這個觀念要幫我們回答什麼？

**核心問題：平均值和標準差改變時，資料會怎麼移動、怎麼散開？68/95/99.7 什麼時候能用？**

- **用在流程哪裡**：連續特徵的形狀判讀（身高、考試分數、量測誤差），以及 GaussianNB 等模型的假設檢查。
- **讀完要能做到**：能讀懂 μ、σ 與區間機率，知道標準化只改尺度、不會把偏態變常態。
- **閱讀方式**：每個程式格前先讀「為什麼做／怎麼看」，再執行；用實際輸出回答「先想一想」。

觀念連結：[特徵工程](../名詞解釋/04_特徵工程/README.md)、[常見算法](../名詞解釋/07_常見算法/README.md)

# 05｜常態分布：中心與分散各由誰決定？

常態（高斯）分布 $N(\mu, \sigma^2)$ 的鐘形曲線由兩個參數完全決定，PDF 為 $f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-(x-\mu)^2 / 2\sigma^2}$。

**一、μ 管位置，σ 管寬窄（分開看才不會混）**

- **平均值 μ**：曲線的中心，對稱軸。μ 從 0 移到 3，整座山平移，形狀不變。
- **標準差 σ**：曲線的寬窄。σ 越大山越矮越寬（但**總面積永遠是 1**）；千萬別把「PDF 比較高」誤讀成「樣本比較多」。
- 本篇左圖固定 σ 只改 μ、右圖固定 μ 只改 σ，一次只動一個參數，這是讀參數圖的正確姿勢。

**二、68／95／99.7：常態假設下的理論承諾**

$P(\mu\pm\sigma) \approx 68\%$、$\mu\pm 2\sigma \approx 95\%$、$\mu\pm 3\sigma \approx 99.7\%$。注意三件事：(1) 這只在**真是常態**時成立；(2) 本篇用假想身高 $N(170, 8^2)$ 把理論機率和抽樣比例並排放——樣本有波動是正常的；(3) **8 是標準差，不是變異數**（變異數是 64）。

**三、標準化：換尺不換形**

Z-score $z = \frac{x - \bar{x}}{s}$ 讓平均變 0、標準差變 1，但它是**線性縮放**，右偏的配送時間標準化後長尾還在。口訣：「平均 0、標準差 1 $\neq$ 常態」，形狀要用直方圖與 Q-Q 圖（見 07）檢查。

**四、接回 GaussianNB 的正確假設**：它假設的是「**給定類別後，各連續特徵的條件分布**」為常態（例如「是籃球隊的身高」常態），不是全部資料混在一起常態；機器學習也不要求所有特徵都常態，要不要轉換看模型假設與驗證結果。

## 準備共用繪圖設定

**為什麼做**：統一繁體中文字型與配色，讓讀圖重點清楚。

**怎麼看**：這格只載入工具，沒有訓練模型；會列出實際字型。需要套件時由老師協助準備 NumPy、pandas、SciPy、Matplotlib、Seaborn。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

# 設定中文字型支援
# --- 設定 Matplotlib 中文字型支援（Mac / Windows / Colab 通用）---
import glob
import os
try:
    import mplfonts
except ImportError:  # Colab 環境若缺套件則自動安裝（本機已在 requirements.txt）
    %pip install -q mplfonts
    import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)   # 直接註冊 mplfonts 內附的字型，不改任何檔案
use_font("Noto Sans CJK SC")              # 套用思源黑體，負號也會一起處理

sns.set_theme(style='whitegrid', rc={
    'axes.unicode_minus': False, 'figure.facecolor': '#F5F7F2',
    'axes.facecolor': '#FFFFFF', 'axes.spines.top': False,
    'axes.spines.right': False, 'grid.alpha': 0.22,
    'axes.titlesize': 14, 'axes.labelsize': 11, 'figure.dpi': 110})
GREEN, ORANGE, PURPLE = '#2F6657', '#C17B45', '#786497'
rng = np.random.default_rng(42)
print('中文字型：Noto Sans CJK SC（mplfonts 通用寫法）')

## 01｜平均值搬動中心，標準差改變寬度

**為什麼做**：分開比較位置與分散，避免把較高的 PDF 誤解為樣本一定較多。

**怎麼看**：左圖固定 σ、只改 μ；右圖固定 μ、只改 σ。所有理論 PDF 在完整實數範圍下的面積都是 1。

In [ ]:
x = np.linspace(-6, 8, 700)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for mu, color in [(0, GREEN), (3, ORANGE)]:
    axes[0].plot(x, stats.norm.pdf(x, loc=mu, scale=1), color=color, label=f'平均值 {mu}')
for sigma, color in [(1, GREEN), (2, PURPLE)]:
    axes[1].plot(x, stats.norm.pdf(x, scale=sigma), color=color, label=f'標準差 {sigma}')
for ax, title in zip(axes, ['平均值決定中心位置', '標準差決定分散程度']):
    ax.set(title=title, xlabel='數值', ylabel='機率密度')
    ax.legend()
plt.show()

## 02｜68／95／99.7 是假設下的理論範圍

**為什麼做**：以假想身高分布 N(170, 8²) 為例，將抽樣比例與理論機率放在一起。

**怎麼看**：這些比例只適用於常態假設；樣本有波動，不預設每群真實身高都常態。8 是標準差，不是變異數。

In [ ]:
mu, sigma = 170, 8
heights = np.random.default_rng(42).normal(mu, sigma, size=4000)
rows = []
for width in [1, 2, 3]:
    a, b = mu - width * sigma, mu + width * sigma
    rows.append({'範圍': f'平均值±{width}個標準差', '下界': a, '上界': b,
                 '理論機率': stats.norm.cdf(b, mu, sigma) - stats.norm.cdf(a, mu, sigma),
                 '觀察比例': np.mean((heights >= a) & (heights <= b))})
display(pd.DataFrame(rows).round(4))
x = np.linspace(138, 202, 500)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(heights, bins=35, density=True, color=GREEN, alpha=.25)
ax.plot(x, stats.norm.pdf(x, mu, sigma), color=GREEN)
mask = (x >= 162) & (x <= 178)
ax.fill_between(x[mask], stats.norm.pdf(x[mask], mu, sigma), color=ORANGE, alpha=.3)
ax.set(title='假想身高：平均值±1個標準差', xlabel='身高（公分）', ylabel='機率密度')
plt.show()

## 03｜標準化改尺度，不保證改形狀

**為什麼做**：把明顯右偏的配送時間做 Z-score，確認線性縮放沒有變出常態形狀。

**怎麼看**：標準化後平均約 0、標準差約 1，但長右尾仍存在。兩圖的橫軸單位不同，密度高度也會改變。

In [ ]:
times = np.random.default_rng(42).lognormal(mean=3, sigma=.7, size=1500)
z = (times - times.mean()) / times.std(ddof=0)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, values, title, unit in zip(axes, [times, z],
        ['原始時間：右偏', 'Z-score 後：仍右偏'], ['時間（分鐘）', '標準化數值']):
    ax.hist(values, bins=40, density=True, color=GREEN, alpha=.7)
    ax.set(title=title, xlabel=unit, ylabel='密度')
plt.show()
print(f'標準化後平均：{z.mean():.4f}，標準差：{z.std(ddof=0):.4f}')
assert np.isclose(z.mean(), 0, atol=1e-12) and np.isclose(z.std(), 1)

## 接回機器學習

GaussianNB 假設的是「給定類別後，各連續特徵的條件分布」為常態，不是要求全部資料混在一起常態。一般機器學習也不要求所有特徵常態；用模型假設與驗證結果決定是否轉換。

## 換你解讀

1. 同樣平均，較大的 σ 會改變什麼？
2. 資料平均為 0、標準差為 1，就一定常態嗎？
3. 應看整體資料，還是各類別內的分布，才能討論 GaussianNB 假設？

### 請 AI 幫你做一個小實驗

請 AI 固定平均 170，將標準差從 8 改成 12，比較 162～178 公分的理論機率。先判斷機率方向再計算。

先說出預期，再看實際圖形，最後寫下「證據 → 解釋 → 不能下的結論」。

[回章節首頁](./README.md)